# toolbar_state

> Centralized client-side toolbar state restoration after HTMX settles

The shared toolbar in the dual-column step is re-rendered by many operations:
chrome switches, mutations (split/merge/undo/reset), NLTK Split, FA toggle,
FA completion, and seg init. Each re-render produces fresh HTML with default
styling, losing any client-side toggle/button state.

This module provides a single `htmx:afterSettle` handler that restores all
client-side toolbar state after any re-render. Adding a new client-side toggle
means adding its restore logic to `_build_restore_entries()` — no changes needed
at any of the toolbar render call sites.

In [ ]:
#| default_exp components.toolbar_state

In [ ]:
#| export
from fasthtml.common import Script

from cjm_transcript_segment_align.components.sync_controls import (
    SYNC_KEY, SYNC_BTN_ID,
)
from cjm_transcript_vad_align.components.audio_controls import (
    AlignAudioControlIds, _TOGGLE_BG_OFF, _TOGGLE_BG_ON,
)
from cjm_transcript_vad_align.components.callbacks import ALIGN_AUDIO_CONFIG

## Toolbar Restore Handler

Generates a standalone JS IIFE that registers an `htmx:afterSettle` handler.
On each settle, it checks whether the relevant toolbar elements exist in the DOM
and restores their styling from client-side JS state.

### Adding a new toolbar toggle

1. Add a restore entry to `_build_restore_entries()` with the element ID,
   state source (JS expression), and class toggle logic.
2. No changes needed at render call sites — the settle handler picks it up automatically.

### Cleanup

Uses the keyed `window` reference pattern to remove the previous handler on re-render.

In [ ]:
#| export
# Unique key for the settle handler on window (for cleanup)
_TOOLBAR_RESTORE_KEY = "_sdToolbarRestoreHandler"


def generate_toolbar_restore_js() -> Script:  # Script element with settle handler
    """Generate the centralized toolbar state restore settle handler.
    
    Restores client-side state for all toolbar toggles/buttons after any
    HTMX settle that re-renders the toolbar. Runs once per settle event.
    """
    sk = ALIGN_AUDIO_CONFIG.state_key
    toggle_id = AlignAudioControlIds.AUTO_NAV_TOGGLE

    return Script(f"""
    (function() {{
        // Clean up previous handler
        if (window.{_TOOLBAR_RESTORE_KEY}) {{
            document.body.removeEventListener('htmx:afterSettle', window.{_TOOLBAR_RESTORE_KEY});
        }}

        function restoreToolbarState() {{
            // --- Sync button: btn-primary (on) / btn-outline (off) ---
            var syncBtn = document.getElementById('{SYNC_BTN_ID}');
            if (syncBtn && window.{SYNC_KEY}) {{
                var syncOn = window.{SYNC_KEY}.enabled || false;
                syncBtn.classList.toggle('btn-primary', syncOn);
                syncBtn.classList.toggle('btn-outline', !syncOn);
            }}

            // --- Auto-play toggle: checked + bg-success (on) / bg-error (off) ---
            var autoToggle = document.getElementById('{toggle_id}');
            if (autoToggle && window.{sk}) {{
                var autoOn = window.{sk}.autoNavigate || false;
                autoToggle.checked = autoOn;
                autoToggle.classList.remove('{_TOGGLE_BG_OFF}', '{_TOGGLE_BG_ON}');
                autoToggle.classList.add(autoOn ? '{_TOGGLE_BG_ON}' : '{_TOGGLE_BG_OFF}');
            }}
        }}

        window.{_TOOLBAR_RESTORE_KEY} = function(evt) {{
            restoreToolbarState();
        }};
        document.body.addEventListener('htmx:afterSettle', window.{_TOOLBAR_RESTORE_KEY});
    }})();
    """)

## Tests

In [ ]:
from fasthtml.common import to_xml

script = generate_toolbar_restore_js()
html = to_xml(script)

# Verify sync button restore
assert SYNC_BTN_ID in html
assert SYNC_KEY in html
assert 'btn-primary' in html
assert 'btn-outline' in html

# Verify auto-play toggle restore
assert AlignAudioControlIds.AUTO_NAV_TOGGLE in html
assert _TOGGLE_BG_OFF in html  # bg-error
assert _TOGGLE_BG_ON in html   # bg-success
assert 'autoNavigate' in html

# Verify cleanup pattern
assert 'removeEventListener' in html
assert _TOOLBAR_RESTORE_KEY in html
assert 'afterSettle' in html

print('Toolbar state tests passed')

Toolbar state tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()